# 02 — Feature Importance Analysis

This notebook performs perturbation-based feature importance on the live autoencoder model.

**Method**: For each of the 19 features, we perturb it by ±1 std dev and measure
how much the reconstruction error changes. Features that cause the largest error
delta are the most important to the anomaly detector.

We also map the 19 live features to the 192-feature research schema in `ML_IOT_PART/featurelist.md`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'python-backend'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0a0e14'
matplotlib.rcParams['axes.facecolor'] = '#12161f'
matplotlib.rcParams['text.color'] = '#e6edf3'
matplotlib.rcParams['axes.labelcolor'] = '#8b949e'
matplotlib.rcParams['xtick.color'] = '#8b949e'
matplotlib.rcParams['ytick.color'] = '#8b949e'

from ml_engine import MLEngine, FEATURE_NAMES, FEATURE_DIM

In [ ]:
MODEL_PATH = os.path.join('..', 'python-backend', 'model.pkl')
engine = MLEngine(input_dim=FEATURE_DIM)

if os.path.exists(MODEL_PATH):
    engine.load(MODEL_PATH)
    print(f'Loaded model. Status: {engine.get_status()}')
else:
    print('No model.pkl found. Train the model first.')

## Feature Schema Mapping

The live autoencoder uses 19 features. The research pipeline (`ML_IOT_PART/featurelist.md`) uses 192.
Here is the mapping between the two schemas.

In [ ]:
schema_mapping = {
    'packet_length': 'pck_size',
    'ip_header_length': 'IP_ihl × 4',
    'ttl': 'IP_ttl',
    'ip_protocol': 'IP_proto',
    'is_ipv6': '(derived — not in research schema)',
    'src_port': 'sport_bare',
    'dst_port': 'dport_bare',
    'tcp_flags': 'TCP_flags',
    'tcp_window_size': 'TCP_window',
    'tcp_header_length': 'TCP_dataofs × 4',
    'udp_length': 'UDP_len',
    'icmp_type': 'ICMP_type',
    'icmp_code': 'ICMP_code',
    'payload_length': 'payload_bytes',
    'flow_packet_count': '(derived — expanding window counter)',
    'flow_byte_count_avg': 'pck_size_mean_WE (approximate)',
    'flow_inter_arrival_ms': 'ts_diff (approximate)',
    'flow_recent_size_std': 'pck_size_std_6 (approximate, window=20)',
    'flow_recent_port_diversity': 'dst_port_diversity (approximate)',
}

print(f"{'Live Feature':<30} {'Research Schema Equivalent'}")
print('=' * 70)
for live, research in schema_mapping.items():
    print(f'{live:<30} {research}')

## Perturbation-Based Feature Importance

In [ ]:
if engine._model is not None and engine._scaler._fitted:
    rng = np.random.default_rng(42)
    # Create a baseline sample at the learned mean
    baseline = engine._scaler.mean_.copy()
    baseline_score = engine.score_packet(baseline)
    
    importance = []
    for i in range(FEATURE_DIM):
        perturbed = baseline.copy()
        perturbed[i] += engine._scaler.std_[i] * 2.0  # perturb by +2σ
        perturbed_score = engine.score_packet(perturbed)
        delta = abs(perturbed_score - baseline_score)
        importance.append(delta)
    
    importance = np.array(importance)
    # Normalize
    if importance.max() > 0:
        importance_norm = importance / importance.max()
    else:
        importance_norm = importance
    
    # Sort by importance
    sorted_idx = np.argsort(importance_norm)[::-1]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    y_pos = np.arange(FEATURE_DIM)
    colors = ['#f85149' if v > 0.7 else '#f0883e' if v > 0.4 else '#58a6ff' for v in importance_norm[sorted_idx]]
    
    ax.barh(y_pos, importance_norm[sorted_idx], color=colors, height=0.6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([FEATURE_NAMES[i] for i in sorted_idx], fontsize=9)
    ax.set_xlabel('Normalized Importance (Δ reconstruction error)')
    ax.set_title('Feature Importance via Perturbation Analysis', fontsize=14, pad=12)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.1)
    
    plt.tight_layout()
    plt.show()
    
    print('\nTop 5 most important features:')
    for rank, idx in enumerate(sorted_idx[:5]):
        print(f'  {rank+1}. {FEATURE_NAMES[idx]}: {importance_norm[idx]:.4f}')
else:
    print('Model not loaded.')